# 01 - Preparação das bases

Baixa e confere SH17 e Construction-PPE, grava as cópias redimensionadas, procura
duplicatas, congela as divisões e confere os YAMLs de E1 e E2
([detalhamento](../docs/fases/fase-2.md)).

**No Colab, sem GPU** (a preparação completa baixa cerca de 15 GB):

1. Execute tudo e autorize o Google Drive. A execução **para** na célula da revisão visual.
2. Revise as folhas de contato, preencha `revisao.csv` no Drive e defina `LIMIAR_APROVADO`.
3. Execute a partir daquela célula (**Ambiente de execução > Executar célula e as seguintes**).

Se a sessão cair, execute tudo de novo: o download é retomado e, depois da primeira
preparação, as bases são restauradas do pacote no Drive. As decisões já registradas ficam.

Fora do Colab o padrão é o modo **curto**: Construction-PPE completa e uma amostra do SH17.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/Guilh3rme/epi-sistema.git"

if "COLAB_RELEASE_TAG" in os.environ:
    raiz = Path("/content/epi-sistema")
    if not raiz.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(raiz)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))  # dispensa reiniciar o ambiente
else:
    raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

(raiz / "runs").mkdir(exist_ok=True)
os.chdir(raiz / "runs")  # pesos e resultados ficam em runs/, fora do git
print("raiz:", raiz.name)

In [ ]:
from epi.ambiente import configurar_ultralytics, em_colab, registrar

configurar_ultralytics(raiz)  # antes de qualquer importação do ultralytics
from epi import dados

# "curto": teste local (Construction-PPE e amostra do SH17). "completo": a preparação da fase.
MODO = os.environ.get("EPI_MODO") or ("completo" if em_colab() else "curto")
assert MODO in ("curto", "completo"), MODO
AMOSTRA_SH17 = {"train": 40, "val": 16}  # só no modo curto

cfg = dados.carregar_config(raiz / "configs" / "dados.yaml")
SEMENTE = cfg["semente"]
COMPLETO = MODO == "completo"
downloads = raiz / "data" / "downloads"
preparado = raiz / "data" / ("preparado" if COMPLETO else "preparado_curto")
saida = raiz / "runs" / ("f2" if COMPLETO else "f2_curto")
pacote = None  # cópia das bases preparadas no Drive
if COMPLETO and em_colab():
    from google.colab import drive

    drive.mount("/content/drive")
    saida = Path("/content/drive/MyDrive/epi-sistema/f2")
    pacote = saida / "bases_preparadas.tar"
divisoes = raiz / "configs" / "divisoes" if COMPLETO else saida / "divisoes"
saida.mkdir(parents=True, exist_ok=True)
info = registrar(saida / "ambiente", semente=SEMENTE)
print(f"MODO: {MODO} | commit: {info['commit']} | CPUs: {os.cpu_count()}")

## Espaço em disco e retomada

In [ ]:
import json
import shutil
import time


def pronta(base):
    return (preparado / f"{base}.json").exists()


if pacote and pacote.exists() and not (pronta("sh17") and pronta("cppe")):
    inicio = time.time()
    dados.restaurar(pacote, preparado.parent)
    print(f"bases restauradas do Drive em {(time.time() - inicio) / 60:.1f} min")

livre = shutil.disk_usage(raiz).free / 1e9
print(f"disco livre: {livre:.1f} GB")
if pacote:
    print(f"Drive livre: {shutil.disk_usage(saida).free / 1e9:.1f} GB (o pacote ocupa cerca de 3 GB)")
if COMPLETO and not pronta("sh17"):
    parcial = sum(p.stat().st_size for p in downloads.glob("sh17.zip*")) / 1e9
    assert livre + parcial >= cfg["espaco_livre_gb"], f"são necessários {cfg['espaco_livre_gb']} GB livres"

## Construction-PPE

In [ ]:
import zipfile

copia = dict(lado=cfg["copia"]["lado_maior"], qualidade=cfg["copia"]["qualidade_jpeg"])
c = cfg["cppe"]
if not pronta("cppe"):
    inicio = time.time()
    arquivo = dados.baixar(c["url"], downloads / "construction-ppe.zip", c["sha256_zip"])
    print(f"download e SHA-256: {(time.time() - inicio) / 60:.1f} min")
    with zipfile.ZipFile(arquivo) as z:
        itens = dados.itens_cppe(z.namelist())
        contagem = {p: sum(i["parte"] == p for i in itens) for p in {i["parte"] for i in itens}}
        assert contagem == c["imagens"], f"contagem de origem divergente: {contagem}"
        resumo = dados.preparar("cppe", itens, z.read, preparado, c["mapa"], vista_comum="cppe6", **copia)
    print(f"{len(itens)} imagens preparadas em {resumo['segundos']:.0f} s")

## SH17

Acesso sem credenciais. Se o Kaggle recusar a partir do Colab, pare e avise: o plano B do
detalhamento (`kagglehub` com o segredo `KAGGLE_API_TOKEN`) ainda não está implementado.

In [ ]:
s = cfg["sh17"]
listas_origem = {}
for nome, esperado in s["sha256_listas"].items():
    arquivo = downloads / "sh17" / nome
    if not arquivo.exists():
        arquivo.parent.mkdir(parents=True, exist_ok=True)
        arquivo.write_bytes(dados.baixar_avulso(f"{s['url']}/{nome}"))
    assert dados.impressao(arquivo) == esperado, f"{nome}: impressão digital divergente"
    listas_origem[nome.split("_")[0]] = dados.ler_lista(arquivo.read_bytes())

contagem = {p: len(x) for p, x in listas_origem.items()}
assert contagem == s["imagens"], f"contagem de origem divergente: {contagem}"
assert not set(listas_origem["train"]) & set(listas_origem["val"])
print("listas do SH17 conferidas:", contagem)

In [ ]:
import hashlib
import random
from concurrent.futures import ThreadPoolExecutor

zip_sh17 = downloads / "sh17.zip"
if COMPLETO and not pronta("sh17"):
    inicio = time.time()
    dados.baixar(s["url"], zip_sh17, s["md5_zip"], "md5")
    print(f"download e MD5: {(time.time() - inicio) / 60:.1f} min")
    itens = dados.itens_sh17(listas_origem["train"], listas_origem["val"])
    with zipfile.ZipFile(zip_sh17) as z:  # lido sem extrair
        for nome, esperado in s["sha256_listas"].items():
            assert hashlib.sha256(z.read(nome)).hexdigest() == esperado, f"{nome} no zip diverge"
        resumo = dados.preparar("sh17", itens, z.read, preparado, s["mapa"], **copia)
    print(f"{len(itens)} imagens preparadas em {resumo['segundos'] / 60:.1f} min")
elif not COMPLETO:
    amostra = raiz / "data" / "amostra_sh17"
    sorteio = random.Random(SEMENTE)
    escolha = {p: sorted(sorteio.sample(listas_origem[p], n)) for p, n in AMOSTRA_SH17.items()}
    itens = dados.itens_sh17(escolha["train"], escolha["val"])

    def trazer(caminho):
        destino = amostra / caminho
        if not destino.exists():
            destino.parent.mkdir(parents=True, exist_ok=True)
            parcial = destino.with_name(destino.name + ".parte")
            parcial.write_bytes(dados.baixar_avulso(f"{s['url']}/{caminho.replace('/', '%2F')}"))
            parcial.replace(destino)

    inicio = time.time()
    with ThreadPoolExecutor(4) as grupo:
        list(grupo.map(trazer, [i[k] for i in itens for k in ("imagem", "rotulo", "voc", "meta")]))
    print(f"amostra obtida em {time.time() - inicio:.0f} s")
    resumo = dados.preparar("sh17", itens, lambda p: (amostra / p).read_bytes(), preparado, s["mapa"], **copia)
    print(f"{len(itens)} imagens preparadas em {resumo['segundos']:.0f} s")

## Pacote das bases no Drive

Gravado logo após a preparação, para que uma sessão perdida não repita o download.

In [ ]:
if pacote and not pacote.exists():
    inicio = time.time()
    dados.empacotar(preparado, pacote)
    print(f"pacote no Drive: {pacote.stat().st_size / 1e9:.2f} GB em {(time.time() - inicio) / 60:.1f} min")

## Conferências

Interrompem: contagem de imagens ou rótulos, dimensões diferentes do VOC, rótulo fora da
faixa (o Ultralytics descartaria a imagem) e imagem preparada ilegível. As contagens de
caixas são comparadas com o artigo do SH17 e o manifesto da Construction-PPE, sem interromper.

In [ ]:
resumos = {b: json.loads((preparado / f"{b}.json").read_text(encoding="utf-8")) for b in ("sh17", "cppe")}
assert resumos["cppe"]["imagens"] == c["imagens"], resumos["cppe"]["imagens"]
esperadas = s["imagens"] if COMPLETO else AMOSTRA_SH17
assert resumos["sh17"]["imagens"] == esperadas, resumos["sh17"]["imagens"]
assert resumos["sh17"]["rotulos"] == sum(esperadas.values()), "rótulos do SH17 faltando"
for base, r in resumos.items():
    print(f"{base}: imagens {r['imagens']}, rótulos {r['rotulos']}")
    assert not r["voc_divergentes"], f"dimensões diferentes do VOC: {r['voc_divergentes'][:20]}"
    assert not r["rotulos_fora_da_faixa"], f"rótulos fora da faixa: {r['rotulos_fora_da_faixa'][:20]}"

nativas = {p: sum(v.values()) for p, v in resumos["cppe"]["caixas_nativas"].items()}
print("Construction-PPE, caixas na origem:", nativas, "| manifesto:", c["caixas_manifesto"])
if COMPLETO:
    print("SH17, caixas por classe do projeto (contadas x artigo):")
    for k, artigo in s["caixas_artigo"].items():
        n = sum(v.get(str(k), 0) for v in resumos["sh17"]["caixas_projeto"].values())
        print(f"  {k}: {n} x {artigo} ({n - artigo:+d})")
print("SH17, fotógrafos (só contagens):", resumos["sh17"].get("fotografos"))

manifesto = dados.ler_manifesto(preparado)
inicio = time.time()
total = dados.conferir_preparado(preparado, manifesto)
print(f"{total} imagens preparadas lidas em {time.time() - inicio:.0f} s")

In [ ]:
if COMPLETO:
    zip_sh17.unlink(missing_ok=True)  # só depois das conferências

## Origem das imagens da Construction-PPE

Contagem por família de nome e folhas de uma amostra de cada família. A fonte não
identifica quais imagens são geradas: o que o autor inspecionar (folhas em `f2/familias/`) é
declarado como estimativa.

In [ ]:
from collections import Counter

from IPython.display import Image as Figura, display

cppe = {k: m for k, m in manifesto.items() if m["base"] == "cppe"}
familias = {}
for chave, m in cppe.items():
    familias.setdefault(dados.familia(chave, c["familias"]), []).append(chave)
resumo_familias = {}
for nome, chaves in sorted(familias.items()):
    formatos = Counter(f"{cppe[k]['largura']}x{cppe[k]['altura']}" for k in chaves)
    resumo_familias[nome] = {
        "imagens": dict(Counter(cppe[k]["parte"] for k in chaves)),
        "formatos_mais_comuns": dict(formatos.most_common(6)),
    }
    print(nome, resumo_familias[nome])
    escolhidas = random.Random(SEMENTE).sample(sorted(chaves), min(c["amostra_por_familia"], len(chaves)))
    folhas = dados.folhas_de_contato(
        [(preparado / "cppe" / "images" / cppe[k]["nome"], cppe[k]["nome"]) for k in sorted(escolhidas)],
        saida / "familias", nome.strip("-"), colunas=5, linhas=10, lado=320)
    print("  folhas:", [f.name for f in folhas])

## Duplicatas

pHash de 64 bits em todos os pares. Até `duplicata_ate`: duplicata; até `revisao_ate`:
revisão visual; acima, até `precaucao_ate`: duplicata
por precaução, sem revisão; acima disso: distinta.

In [ ]:
import matplotlib.pyplot as plt

limiares = cfg["phash"]
inicio = time.time()
pares, histograma = dados.achar_pares(manifesto, limiares["precaucao_ate"])
print(f"{len(manifesto)} imagens, {histograma.sum():,} pares em {time.time() - inicio:.1f} s")
print("pares por distância:", {d: int(histograma[d]) for d in range(limiares["precaucao_ate"] + 1)})
print("arquivos idênticos (SHA-256):", sum(p["identico"] for p in pares))

figura, eixos = plt.subplots(1, 2, figsize=(11, 3))
for eixo, ate, titulo in ((eixos[0], 64, "todos os pares"), (eixos[1], 16, "até 16 bits")):
    eixo.bar(range(ate + 1), histograma[: ate + 1])
    eixo.set(title=titulo, xlabel="distância de Hamming")
eixos[0].set_yscale("log")
figura.tight_layout()
figura.savefig(saida / "histograma_phash.png", dpi=120)
plt.show()

In [ ]:
def caminho(chave):
    base, nome = chave.split("/", 1)
    return preparado / base / "images" / nome


revisao = saida / "revisao" / "revisao.csv"
faixa = dados.faixa_revisao(pares, limiares["duplicata_ate"], limiares["revisao_ate"])
linhas = dados.gravar_revisao(faixa, revisao)
numero = {(x["a"], x["b"]): int(x["n"]) for x in linhas}
celulas = []
for p in sorted(faixa, key=lambda p: numero[p["a"], p["b"]]):
    for lado in ("a", "b"):
        celulas.append((caminho(p[lado]), f"{numero[p['a'], p['b']]} (d={p['distancia']}) {p[lado]}"))
folhas = dados.folhas_de_contato(celulas, revisao.parent, "pares", colunas=2, linhas=6)
print(f"{len(faixa)} pares para revisar, em {len(folhas)} folhas")
print("preencha a coluna duplicata (sim ou nao) em:", revisao)
for folha in folhas[:20]:  # as demais só no Drive, para o notebook não pesar
    display(Figura(filename=str(folha)))

## PARADA: revisão visual do autor

1. Veja as folhas acima (ou em `epi-sistema/f2/revisao/` no Drive). O número de cada par é a
   coluna `n` do `revisao.csv`.
2. Preencha a coluna `duplicata` com `sim` ou `nao` em todas as linhas e salve **no mesmo
   arquivo** (o separador é `;`).
3. Inspecione o histograma e defina abaixo `LIMIAR_APROVADO = (4, 10)`: até o primeiro número,
   duplicata automática; até o segundo, vale a revisão. Acima dele e até
   `precaucao_ate` (`configs/dados.yaml`), o par é duplicata sem revisão; o resto é distinto. Se houver pares demais,
   o segundo número pode ser reduzido (por exemplo, `(4, 8)`): registre a mudança na fase.
4. Execute esta célula e as seguintes. Com par sem decisão, nada é gerado.

In [ ]:
LIMIAR_APROVADO = None  # <- o autor define depois da revisão, por exemplo (4, 10)

decisoes = dados.ler_decisoes(revisao)
if not COMPLETO:  # teste local: não vale como revisão
    pendentes = [p for p in faixa if (p["a"], p["b"]) not in decisoes]
    decisoes |= {(p["a"], p["b"]): False for p in pendentes}
    LIMIAR_APROVADO = (limiares["duplicata_ate"], limiares["revisao_ate"])
    print("!" * 70)
    print(f"MODO CURTO: {len(pendentes)} pares sem revisão marcados como DISTINTOS só para o teste.")
    print("!" * 70)
assert LIMIAR_APROVADO is not None, "PARADA: revise os pares e defina LIMIAR_APROVADO (ver acima)."
duplicata_ate, revisao_ate = LIMIAR_APROVADO
assert 0 <= duplicata_ate <= revisao_ate <= limiares["revisao_ate"], "a busca não cobre esse limiar"
marcados = dados.decidir(pares, decisoes, duplicata_ate, revisao_ate, limiares["precaucao_ate"])
print("duplicatas:", sum(p["duplicata"] for p in marcados), "de", len(marcados), "pares |",
      "por precaução:", sum(p["origem"] == "precaucao" for p in marcados))

## Divisão e congelamento

In [ ]:
listas, efeitos = dados.dividir(manifesto, marcados, cfg["fracao_desenvolvimento"], SEMENTE, c["familias"])
if not COMPLETO:
    shutil.rmtree(divisoes, ignore_errors=True)  # no teste local nada fica congelado
relatorio = dados.congelar(divisoes, listas, efeitos, marcados, manifesto, {
    "modo": MODO,
    "semente": SEMENTE,
    "fracao_desenvolvimento": cfg["fracao_desenvolvimento"],
    "limiar_aprovado": list(LIMIAR_APROVADO),
    "precaucao_ate": limiares["precaucao_ate"],
    "pares_por_distancia": [int(x) for x in histograma],
    "origem": {"sh17_md5_zip": s["md5_zip"], "sh17_sha256_listas": s["sha256_listas"],
               "cppe_sha256_zip": c["sha256_zip"]},
    "referencia": {"sh17_caixas_artigo": s["caixas_artigo"], "cppe_caixas_manifesto": c["caixas_manifesto"]},
    "bases": {b: {k: v for k, v in r.items() if k != "segundos"} for b, r in resumos.items()},
    "familias_cppe": resumo_familias,
    "commit": info["commit"],
})
dados.listas_ultralytics(divisoes, preparado)  # confere as impressões digitais antes
for nome, r in relatorio["listas"].items():
    print(f"{nome}: {r['imagens']} imagens, caixas {r['caixas']}")
print(json.dumps(relatorio["experimentos"], indent=2))

In [ ]:
partes = {e: {p: len(listas[f"{e}_{p}"]) for p in ("teste", "teste_cruzado")} for e in ("e1", "e2")}
n_sh17_val = relatorio["bases"]["sh17"]["imagens"]["val"]
assert partes == {"e1": {"teste": n_sh17_val, "teste_cruzado": sum(c["imagens"].values())},
                  "e2": {"teste": c["imagens"]["val"], "teste_cruzado": n_sh17_val}}, partes
print("testes completos:", partes)

## YAMLs: `model.val` em poucas imagens de cada teste

Modelo sem treino, só para conferir caminhos, rótulos e número de classes. As métricas
impressas não significam nada e não são usadas.

In [ ]:
import torch

device = 0 if torch.cuda.is_available() else "cpu"
for nome in ("base_e1", "base_e1_cppe", "base_e2", "base_e2_sh17"):
    imagens, caixas = dados.verificar_vista(
        raiz / "configs" / f"{nome}.yaml", preparado, raiz / "runs" / "f2_val", device=device)
    print(f"{nome}: {imagens} imagens e {caixas} caixas lidas")

## Entrega

In [ ]:
if COMPLETO and em_colab():
    shutil.copytree(divisoes, saida / "divisoes", dirs_exist_ok=True)
    arquivo = shutil.make_archive(str(saida / "divisoes"), "zip", divisoes)
    print("traga para configs/divisoes/ do repositório o conteúdo de:", arquivo)
print("divisões em:", divisoes)
print("preparação OK" if COMPLETO else "teste curto OK (as divisões geradas não valem)")